# **Stage 4: Multi-Camera Association**

**Pipeline:** Stage 1 (DeepOC-SORT) → Stage 2 (FastReID Features) → **Stage 4 (Multi-Camera Association)** → Stage 5 (Evaluation)

## **Architecture Overview**

```
Input: Stage 2 features.npz (N tracklets × 2048-D embeddings)
       Stage 1 tracklets.json (per-camera metadata)
  ↓
1. Similarity Computation
   - Cosine similarity (post PCA/whitening from Stage 2)
   - Spatio-temporal constraints (camera adjacency + transition time)
   - Threshold filtering (configurable, default 0.40)
  ↓
2. Graph Construction (NetworkX)
   - Nodes = per-camera tracklets
   - Edges = high similarity + valid transitions
  ↓
3. Connected Components Clustering
   - Unsupervised: no training needed
   - Interpretable: visualize similarity graph
  ↓
4. Global Trajectory Reconstruction
   - Assign global IDs to connected components
   - Merge per-camera trajectories into unified timelines
   - Generate outputs for Stage 5 evaluation
  ↓
Output: global_tracklets.json (multi-camera trajectories)
        similarity_graph.graphml (for inspection)
```

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELL 1: IMPORTS & ENVIRONMENT CHECK
# ═══════════════════════════════════════════════════════════

import numpy as np
import json
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Set, Optional
from collections import defaultdict

# Graph processing
import networkx as nx

# Progress bars
from tqdm.auto import tqdm

# Visualization
import matplotlib.pyplot as plt

print("═" * 70)
print("  STAGE 4: MULTI-CAMERA ASSOCIATION")
print("═" * 70)
print(f"NumPy: {np.__version__}")
print(f"NetworkX: {nx.__version__}")
print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 2 — UNZIP & SET ALL PATHS
# ═══════════════════════════════════════════════════════════════════════
import zipfile, os, json
from pathlib import Path

# ── Stage 2 zip (contains features + stage1unzipped inside) ──────────
S2_ZIP     = Path("/kaggle/input/notebooks/ali369/gp-stage-2/_output_.zip")
S2_EXTRACT = Path("/kaggle/working/stage2unzipped")

if not S2_ZIP.exists():
    raise FileNotFoundError(f"❌ Not found: {S2_ZIP}")

if not S2_EXTRACT.exists():
    print(f"Extracting {S2_ZIP} → {S2_EXTRACT} ...")
    S2_EXTRACT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(S2_ZIP, 'r') as zf:
        zf.extractall(S2_EXTRACT)
    print("Done!")
else:
    print("Already extracted, skipping.")

# ── Find files wherever they landed inside the zip ───────────────────
all_tracklets = list(S2_EXTRACT.rglob("tracklets.json"))
all_crops     = list(S2_EXTRACT.rglob("crops"))
all_features  = list(S2_EXTRACT.rglob("reid_features_improved.npz"))

TRACKLETS_JSON = all_tracklets[0] if all_tracklets else None
CROPS_ROOT     = all_crops[0]     if all_crops     else None
FEATURES_NPZ   = all_features[0]  if all_features  else None
OUTPUT_DIR     = Path("/kaggle/working/stage4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "="*60)
print("  PATHS RESOLVED")
print("="*60)
print(f"  tracklets.json : {TRACKLETS_JSON}")
print(f"  crops/         : {CROPS_ROOT}")
print(f"  features.npz   : {FEATURES_NPZ}")
print(f"  output/        : {OUTPUT_DIR}")
print("="*60)

if not TRACKLETS_JSON:
    raise FileNotFoundError("tracklets.json not found inside zip!")
if not FEATURES_NPZ:
    raise FileNotFoundError("reid_features_improved.npz not found inside zip!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — CONFIG + LOAD DATA  [IMPROVED]
# ═══════════════════════════════════════════════════════════════════════
import numpy as np, json, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from collections import defaultdict
import networkx as nx
from tqdm.auto import tqdm

@dataclass
class AssociationConfig:
    similarity_threshold: float = 0.55   # lowered slightly for better recall (was 0.60)
    top_k_candidates:     int   = 10     # increased for better coverage (was 5)
    batch_size:           int   = 5000
    min_component_size:   int   = 1
    max_component_size:   int   = 12     # relaxed slightly (was 8)
    use_spatial_constraints:  bool = False
    use_temporal_constraints: bool = True   # ← ENABLED (was False)
    min_transition_time: float = -300.0    # allow 5-min overlap for adjacent cameras
    max_transition_time: float = 3600.0
    camera_topology: Dict = field(default_factory=dict)
    # ── NEW ──────────────────────────────────────────────────────────────
    scenario_blocking:   bool  = True    # never link tracklets across scenarios
    use_mutual_topk:     bool  = True    # require reciprocal top-K agreement
    min_tracklet_frames: int   = 3       # drop tracklets shorter than this
    length_weight_power: float = 0.5     # edge weight = sim * geometric_mean(n_frames)^p
    bridge_prune_margin: float = 0.05    # prune bridges below threshold + margin

CFG = AssociationConfig()

@dataclass
class Paths:
    tracklets_json:  Path = TRACKLETS_JSON
    features_npz:    Path = FEATURES_NPZ
    crops_root:      Path = CROPS_ROOT
    output_dir:      Path = OUTPUT_DIR
    global_tracklets: Path = OUTPUT_DIR / "global_tracklets.json"
    similarity_graph: Path = OUTPUT_DIR / "similarity_graph.graphml"

PATHS = Paths()

# ── Load raw data ────────────────────────────────────────────────────────
print(f"Loading features from {PATHS.features_npz}...")
data         = np.load(PATHS.features_npz, allow_pickle=True)
features_raw = data['features'].astype(np.float32)
meta_raw     = data['metadata']
metadata_all = meta_raw.item() if (isinstance(meta_raw, np.ndarray) and meta_raw.ndim==0) \
               else list(meta_raw)

print(f"Loading tracklets from {PATHS.tracklets_json}...")
with open(PATHS.tracklets_json, 'r') as f:
    stage1_tracklets = json.load(f)

# ── Build lookup dicts ───────────────────────────────────────────────────
tracklet_cameras = {}
tracklet_times   = {}
for m in metadata_all:
    tid = m['tracklet_id']
    tracklet_cameras[tid] = (m.get('scenario', 'unknown'), str(m.get('camera_id', 'unknown')))
for tid, tdata in stage1_tracklets.items():
    frames = tdata.get('frames', [])
    if frames:
        tracklet_times[tid] = (
            float(tdata.get('start_time', 0.0)),
            float(tdata.get('end_time', float(len(frames))))
        )

# ── Filter short tracklets  (accuracy: remove noisy 1-2 frame tracklets) ─
n_frames_map = {m['tracklet_id']: m.get('n_frames', 0) for m in metadata_all}
keep_mask = np.array(
    [n_frames_map.get(m['tracklet_id'], 0) >= CFG.min_tracklet_frames for m in metadata_all],
    dtype=bool
)
features = features_raw[keep_mask]
metadata = [m for m, k in zip(metadata_all, keep_mask) if k]
print(f"  Dropped {keep_mask.size - keep_mask.sum()} short tracklets (<{CFG.min_tracklet_frames} frames)")

# ── Sanitize & re-normalize ───────────────────────────────────────────────
bad = np.sum(~np.isfinite(features))
if bad > 0:
    features = np.nan_to_num(features, nan=0.0, posinf=1.0, neginf=-1.0)
norms    = np.linalg.norm(features, axis=1, keepdims=True)
features = features / np.maximum(norms, 1e-8)

# ── Pre-encode cameras & scenarios as int arrays (for vectorized masking) ─
_cam_list  = [tracklet_cameras.get(m['tracklet_id'], ('?','?')) for m in metadata]
_scen_list = [c[0] for c in _cam_list]
unique_cams  = {c: i for i, c in enumerate(set(_cam_list))}
unique_scens = {s: i for i, s in enumerate(set(_scen_list))}
cam_ints     = np.array([unique_cams[c]  for c in _cam_list],  dtype=np.int32)
scen_ints    = np.array([unique_scens[s] for s in _scen_list], dtype=np.int32)
nframes_arr  = np.array([n_frames_map.get(m['tracklet_id'], 1) for m in metadata], dtype=np.float32)

# ── Time array shape (N, 2) ───────────────────────────────────────────────
N = len(metadata)
time_arr = np.zeros((N, 2), dtype=np.float32)
for i, m in enumerate(metadata):
    tid = m['tracklet_id']
    if tid in tracklet_times:
        time_arr[i] = tracklet_times[tid]

print(f"\n{'='*60}")
print(f"  CFG   : threshold={CFG.similarity_threshold}, top_k={CFG.top_k_candidates}")
print(f"  Data  : {N} tracklets × {features.shape[1]}-D")
print(f"  Cams  : {len(unique_cams)} unique (scenario, camera) pairs")
print(f"  Scens : {len(unique_scens)} scenarios")
print(f"{'='*60}")
print("✅ Ready for Cell 4")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4: SIMILARITY COMPUTATION — FULLY VECTORIZED  [IMPROVED]
# ═══════════════════════════════════════════════════════════════════════

def compute_similarity_vectorized(
    features:   np.ndarray,
    cam_ints:   np.ndarray,
    scen_ints:  np.ndarray,
    time_arr:   np.ndarray,
    cfg:        AssociationConfig,
    batch_size: int = 5000
) -> np.ndarray:
    """
    Compute pairwise cosine similarity with ALL constraints applied
    inside the batch loop via NumPy broadcasting — no Python inner loop.
    """
    N = features.shape[0]
    similarity = np.zeros((N, N), dtype=np.float32)

    print(f"Computing similarity for {N} tracklets (batch={batch_size})...")
    t0 = time.time()

    for i in tqdm(range(0, N, batch_size), desc="Batch cosine + constraints"):
        end_i = min(i + batch_size, N)
        B     = end_i - i
        batch = features[i:end_i]            # (B, D)
        sims  = np.dot(batch, features.T)    # (B, N)  — cosine sim (already L2-normed)

        # 1. Zero self-similarity (diagonal)
        for b in range(B):
            sims[b, i + b] = 0.0

        # 2. Block same-camera pairs (vectorized) — replaces old Python inner loop
        same_cam = cam_ints[i:end_i, None] == cam_ints[None, :]   # (B, N) bool
        sims[same_cam] = 0.0

        # 3. Block cross-scenario pairs (NEW — always correct, large accuracy gain)
        if cfg.scenario_blocking:
            diff_scen = scen_ints[i:end_i, None] != scen_ints[None, :]  # (B, N)
            sims[diff_scen] = 0.0

        # 4. Temporal constraint — vectorized (NEW)
        if cfg.use_temporal_constraints:
            t_start = time_arr[:, 0]   # (N,)
            t_end   = time_arr[:, 1]   # (N,)
            # dt_fwd[b,j] = how long after tracklet (i+b) ends does j start
            dt_fwd = t_start[None, :] - t_end[i:end_i, None]   # (B, N)
            dt_bwd = t_start[i:end_i, None] - t_end[None, :]   # (B, N)
            valid_temporal = (
                ((dt_fwd >= cfg.min_transition_time) & (dt_fwd <= cfg.max_transition_time)) |
                ((dt_bwd >= cfg.min_transition_time) & (dt_bwd <= cfg.max_transition_time))
            )
            sims[~valid_temporal] = 0.0

        similarity[i:end_i] = sims

    # Symmetrize (broadcasting can create tiny asymmetries)
    similarity = np.maximum(similarity, similarity.T)
    elapsed    = time.time() - t0
    valid_mask = similarity > 0
    print(f"✓ Done in {elapsed:.1f}s  |  non-zero pairs: {valid_mask.sum()//2:,}")
    print(f"  Mean (valid): {similarity[valid_mask].mean():.4f}  "
          f"Std: {similarity[valid_mask].std():.4f}")
    return similarity

print("═" * 70)
print(" COMPUTING SIMILARITY MATRIX  [VECTORIZED + SCENARIO BLOCKING]")
print("═" * 70)

similarity_matrix = compute_similarity_vectorized(
    features, cam_ints, scen_ints, time_arr, CFG, CFG.batch_size
)
print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5: BUILD SIMILARITY GRAPH  [MUTUAL TOP-K + ARGPARTITION]  [IMPROVED]
# ═══════════════════════════════════════════════════════════════════════

print("═" * 70)
print(" BUILDING SIMILARITY GRAPH  [MUTUAL TOP-K + LENGTH WEIGHTING]")
print("═" * 70)
t0 = time.time()

G = nx.Graph()
for i, m in enumerate(metadata):
    G.add_node(i,
        tracklet_id=m['tracklet_id'],
        scenario=m.get('scenario', 'unknown'),
        camera_id=str(m.get('camera_id', 'unknown')),
        n_frames=m.get('n_frames', 0)
    )

# ── Pre-compute top-K neighbor sets using argpartition O(N) per row ──────
#    argpartition is O(N) vs argsort O(N log N) — meaningful speedup for N≈4000
K = CFG.top_k_candidates
topk_sets = {}

print(f"Computing top-{K} sets with argpartition...")
for i in tqdm(range(N), desc="Top-K sets"):
    row = similarity_matrix[i].copy()
    row[i] = -1.0
    if N > K:
        # argpartition: only partial sort — O(N) instead of O(N log N)
        part      = np.argpartition(row, -K)[-K:]
        top_vals  = row[part]
        valid     = top_vals > 0
        topk_sets[i] = set(part[valid].tolist())
    else:
        topk_sets[i] = set(np.where(row > 0)[0].tolist())

# ── Add edges: threshold + mutual top-K + length weighting ───────────────
print(f"\nAdding edges (threshold={CFG.similarity_threshold}, "
      f"mutual={CFG.use_mutual_topk}, length_power={CFG.length_weight_power})...")

edge_count      = 0
similarity_vals = []

for i in tqdm(range(N), desc="Building graph"):
    for j in topk_sets[i]:
        if j <= i:
            continue                                          # avoid duplicate edges

        sim_val = float(similarity_matrix[i, j])
        if sim_val < CFG.similarity_threshold:
            continue

        # Mutual top-K: j must also have i in its top-K candidates
        # This one check eliminates the majority of false-positive edges
        if CFG.use_mutual_topk and (i not in topk_sets[j]):
            continue

        # Length-weighted similarity: longer tracklets earn higher confidence
        li = float(nframes_arr[i])
        lj = float(nframes_arr[j])
        geom_mean   = (li * lj) ** CFG.length_weight_power
        max_len     = max(li, lj) ** CFG.length_weight_power + 1e-6
        length_w    = geom_mean / max_len           # in (0, 1]
        weighted_sim = sim_val * (0.5 + 0.5 * length_w)

        G.add_edge(i, j, weight=weighted_sim, similarity=sim_val)
        edge_count += 1
        similarity_vals.append(sim_val)

print(f"✓ Graph built in {time.time()-t0:.1f}s")
print(f"  Nodes: {G.number_of_nodes():,}  |  Edges: {G.number_of_edges():,}")
if similarity_vals:
    print(f"  Edge sim: mean={np.mean(similarity_vals):.3f}, "
          f"std={np.std(similarity_vals):.3f}, "
          f"min={np.min(similarity_vals):.3f}, max={np.max(similarity_vals):.3f}")

print(f"\nSaving graph → {PATHS.similarity_graph}...")
nx.write_graphml(G, str(PATHS.similarity_graph))
print("✓ Graph saved")
print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6: CLUSTERING WITH BRIDGE PRUNING + SCENARIO INTEGRITY  [IMPROVED]
# ═══════════════════════════════════════════════════════════════════════

print("═" * 70)
print(" CLUSTERING: BRIDGE PRUNING + SCENARIO INTEGRITY + COMPONENT FILTER")
print("═" * 70)
t0 = time.time()

# ── Step 1: Prune weak bridge edges ──────────────────────────────────────
# A bridge is the single edge holding two sub-graphs together.
# If its similarity is weak (below threshold + margin), removing it is
# much safer than allowing a potential false merge to propagate.
BRIDGE_THRESHOLD = CFG.similarity_threshold + CFG.bridge_prune_margin  # e.g. 0.60

G_pruned       = G.copy()
bridges        = list(nx.bridges(G_pruned))
pruned_bridges = 0
for u, v in bridges:
    if G_pruned[u][v].get('similarity', 1.0) < BRIDGE_THRESHOLD:
        G_pruned.remove_edge(u, v)
        pruned_bridges += 1

print(f"✓ Bridge pruning: removed {pruned_bridges} weak bridge edges "
      f"(threshold={BRIDGE_THRESHOLD:.2f})")

# ── Step 2: Extract components ─────────────────────────────────────────
components_raw = list(nx.connected_components(G_pruned))

# ── Step 3: Scenario integrity — split any component spanning >1 scenario ─
# (This should rarely fire after scenario_blocking in Cell 4, but is a
#  safe fallback in case of any leakage.)
components     = []
scenario_splits = 0
for comp in components_raw:
    scen_groups = defaultdict(set)
    for node in comp:
        scen = G_pruned.nodes[node].get('scenario', 'unknown')
        scen_groups[scen].add(node)
    if len(scen_groups) == 1:
        components.append(comp)
    else:
        for scen_nodes in scen_groups.values():
            components.append(frozenset(scen_nodes))
        scenario_splits += 1

if scenario_splits:
    print(f"✓ Scenario guard: split {scenario_splits} cross-scenario components")

# ── Step 4: Size filter ──────────────────────────────────────────────────
valid_components = [
    c for c in components
    if CFG.min_component_size <= len(c) <= CFG.max_component_size
]
filtered = len(components) - len(valid_components)
if filtered:
    print(f"  Size filter: removed {filtered} components "
          f"(outside [{CFG.min_component_size}, {CFG.max_component_size}])")

print(f"✓ Valid components: {len(valid_components)}  (in {time.time()-t0:.1f}s)")

component_sizes = [len(c) for c in valid_components]
if component_sizes:
    print(f"  Sizes: mean={np.mean(component_sizes):.1f}, "
          f"median={np.median(component_sizes):.0f}, "
          f"max={np.max(component_sizes)}")

# ── Assign global IDs ─────────────────────────────────────────────────────
node_to_global_id = {}
for global_id, comp in enumerate(valid_components, start=1):
    for node_idx in comp:
        node_to_global_id[node_idx] = global_id

next_global_id = len(valid_components) + 1
for i in range(N):
    if i not in node_to_global_id:
        node_to_global_id[i] = next_global_id
        next_global_id += 1

print(f"✓ Assigned {len(set(node_to_global_id.values()))} global track IDs")
print(f"  Multi-camera: {len(valid_components)}")
print(f"  Single-camera: {next_global_id - len(valid_components) - 1}")
print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELL 7: RECONSTRUCT GLOBAL TRAJECTORIES
# ═══════════════════════════════════════════════════════════

print("═" * 70)
print("  RECONSTRUCTING GLOBAL TRAJECTORIES")
print("═" * 70)

# Build global tracklets: global_id -> list of per-camera tracklets
global_tracklets = defaultdict(list)

for node_idx, global_id in node_to_global_id.items():
    tracklet_id = metadata[node_idx]['tracklet_id']
    
    # Get full tracklet data from Stage 1
    if tracklet_id in stage1_tracklets:
        track_data = stage1_tracklets[tracklet_id].copy()
        track_data['local_tracklet_id'] = tracklet_id
        track_data['global_id'] = int(global_id)
        
        # Add camera and scenario info
        scenario, camera = tracklet_cameras.get(tracklet_id, ('unknown', 'unknown'))
        track_data['scenario'] = scenario
        track_data['camera_id'] = str(camera)
        
        # Add temporal info if available
        if tracklet_id in tracklet_times:
            start_time, end_time = tracklet_times[tracklet_id]
            track_data['start_time'] = float(start_time)
            track_data['end_time'] = float(end_time)
            track_data['duration'] = float(end_time - start_time)
        
        global_tracklets[global_id].append(track_data)

print(f"✓ Reconstructed {len(global_tracklets)} global trajectories")

# Sort tracklets within each global ID by start time
for global_id in global_tracklets:
    global_tracklets[global_id].sort(key=lambda x: x.get('start_time', 0.0))

# Compute statistics
multi_camera_count = sum(1 for tracks in global_tracklets.values() if len(tracks) > 1)
single_camera_count = len(global_tracklets) - multi_camera_count

print(f"  Multi-camera trajectories: {multi_camera_count}")
print(f"  Single-camera trajectories: {single_camera_count}")

# Find longest trajectories
max_cameras = max((len(tracks) for tracks in global_tracklets.values()), default=0)
print(f"  Max cameras in single trajectory: {max_cameras}")

# Save global tracklets
print(f"\nSaving global tracklets to {PATHS.global_tracklets}...")

# Convert to JSON-serializable format
output_data = {
    str(global_id): tracks
    for global_id, tracks in global_tracklets.items()
}

with open(PATHS.global_tracklets, 'w') as f:
    json.dump(output_data, f, indent=2)

file_size_mb = PATHS.global_tracklets.stat().st_size / (1024 * 1024)
print(f"✓ Saved global tracklets ({file_size_mb:.1f} MB)")

print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELL 8: VISUALIZATION & ANALYSIS
# ═══════════════════════════════════════════════════════════

print("═" * 70)
print("  VISUALIZATION & ANALYSIS")
print("═" * 70)

# === 1. Component Size Distribution ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Component size histogram
ax = axes[0]
ax.hist(component_sizes, bins=min(30, max(component_sizes)), 
        color='steelblue', edgecolor='black', alpha=0.75)
ax.axvline(np.mean(component_sizes), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {np.mean(component_sizes):.1f}')
ax.set_xlabel('Component Size (# Tracklets)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Connected Component Size Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Plot 2: Similarity distribution of edges
ax = axes[1]
edge_sims = [data['similarity'] for _, _, data in G.edges(data=True)]
if edge_sims:
    ax.hist(edge_sims, bins=50, color='darkorange', edgecolor='black', alpha=0.75)
    ax.axvline(CFG.similarity_threshold, color='red', linestyle='--', 
               linewidth=2, label=f'Threshold: {CFG.similarity_threshold}')
    ax.axvline(np.mean(edge_sims), color='green', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(edge_sims):.3f}')
ax.set_xlabel('Cosine Similarity', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Edge Similarity Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Plot 3: Multi-camera trajectory distribution
ax = axes[2]
camera_counts = defaultdict(int)
for tracks in global_tracklets.values():
    n_cameras = len(tracks)
    camera_counts[n_cameras] += 1

x = sorted(camera_counts.keys())
y = [camera_counts[k] for k in x]
ax.bar(x, y, color='seagreen', edgecolor='black', alpha=0.75)
ax.set_xlabel('# Cameras in Trajectory', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Multi-Camera Trajectory Distribution', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = PATHS.output_dir / 'stage4_analysis.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()

print(f"✓ Saved analysis plots to {fig_path}")

# === 2. Sample Multi-Camera Trajectories ===
print(f"\n{'═' * 70}")
print("  SAMPLE MULTI-CAMERA TRAJECTORIES")
print("═" * 70)

multi_camera_tracks = [
    (gid, tracks) for gid, tracks in global_tracklets.items() if len(tracks) > 1
]

if multi_camera_tracks:
    # Show top 5 longest trajectories
    multi_camera_tracks.sort(key=lambda x: len(x[1]), reverse=True)
    
    for gid, tracks in multi_camera_tracks[:5]:
        print(f"\nGlobal ID {gid}: {len(tracks)} cameras")
        print(f"{'  Camera':<12} {'Start (s)':<12} {'End (s)':<12} {'Duration (s)':<15} {'Frames':<10}")
        print(f"  {'-' * 70}")
        
        for track in tracks:
            cam = f"{track.get('scenario', '?')}-{track.get('camera_id', '?')}"
            start = track.get('start_time', 0.0)
            end = track.get('end_time', 0.0)
            duration = track.get('duration', 0.0)
            nframes = len(track.get('frames', []))
            
            print(f"  {cam:<12} {start:<12.1f} {end:<12.1f} {duration:<15.1f} {nframes:<10}")
else:
    print("No multi-camera trajectories found.")

print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELL 9: FINAL SUMMARY & OUTPUTS
# ═══════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  STAGE 4 COMPLETE - SUMMARY")
print("═" * 70)

print(f"\n📊 INPUT:")
print(f"  Per-camera tracklets: {len(stage1_tracklets)}")
print(f"  Feature vectors: {features.shape[0]} × {features.shape[1]}D")

print(f"\n🔗 GRAPH CONSTRUCTION:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Similarity threshold: {CFG.similarity_threshold}")

print(f"\n🌐 GLOBAL TRAJECTORIES:")
print(f"  Total global IDs: {len(global_tracklets)}")
print(f"  Multi-camera tracks: {multi_camera_count}")
print(f"  Single-camera tracks: {single_camera_count}")
print(f"  Max cameras in trajectory: {max_cameras}")

print(f"\n💾 OUTPUTS:")
print(f"  {PATHS.global_tracklets}")
print(f"  {PATHS.similarity_graph}")
print(f"  {PATHS.output_dir / 'stage4_analysis.png'}")

print(f"\n✅ Stage 4 ready for Stage 5 (Evaluation)")
print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 10 — VIDEO OUTPUT + INLINE DISPLAY (H.264 for browser playback)
# ═══════════════════════════════════════════════════════════════════════
import cv2, numpy as np, subprocess, shutil, base64
from pathlib import Path
from collections import defaultdict
from IPython.display import display, HTML

VIDEO_DIR   = Path("/kaggle/working/stage4/demo_videos")
CROPS_ROOT  = PATHS.crops_root                # ← correct path from Cell 3
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

TOP_N          = 5
FPS_OUT        = 5
PANEL_W        = 280
PANEL_H        = 210
MAX_CAMS_ROW   = 5
FRAMES_PER_CAM = 20
TITLE_H        = 38
TIMELINE_H     = 50
GAP            = 6

COLOURS = [
    (0,220,0),(0,140,255),(255,50,50),(200,0,200),
    (0,220,220),(180,100,0),(0,100,255),(100,200,100),
]

def resolve_crop(frame_str):
    p = CROPS_ROOT / frame_str
    if p.exists(): return p
    matches = list(CROPS_ROOT.rglob(Path(frame_str).name))
    return matches[0] if matches else None

def load_panel(frame_str, border_color, active=True):
    path = resolve_crop(frame_str) if frame_str else None
    if path and path.exists():
        img = cv2.imread(str(path))
        if img is not None:
            img = cv2.resize(img, (PANEL_W, PANEL_H))
            if not active: img = (img * 0.35).astype(np.uint8)
            cv2.rectangle(img, (0,0), (PANEL_W-1,PANEL_H-1), border_color, 4 if active else 1)
            return img
    img = np.zeros((PANEL_H, PANEL_W, 3), dtype=np.uint8); img[:] = (40,40,40)
    cv2.putText(img, "no crop", (PANEL_W//2-30, PANEL_H//2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (120,120,120), 1)
    return img

def put_label(img, text, x, y, color=(255,255,255), scale=0.45, thick=1):
    (tw,th),_ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
    cv2.rectangle(img, (x-2,y-th-3), (x+tw+2,y+3), (0,0,0), -1)
    cv2.putText(img, text, (x,y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick, cv2.LINE_AA)

def render_video(global_id, tracks, out_path_raw):
    n_cams = len(tracks)
    cols   = min(n_cams, MAX_CAMS_ROW)
    rows   = (n_cams + cols - 1) // cols
    cw     = cols*(PANEL_W+GAP)+GAP
    ch     = TITLE_H + rows*(PANEL_H+GAP)+GAP+TIMELINE_H

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path_raw), fourcc, FPS_OUT, (cw, ch))
    if not writer.isOpened(): return False

    all_cams = [t.get("camera_id", f"cam{i}") for i,t in enumerate(tracks)]
    visited  = set()

    for cam_idx, track in enumerate(tracks):
        cam_id  = track.get("camera_id", f"cam{cam_idx}")
        flist   = track.get("frames", [])
        t_start = track.get("start_time", 0.0)
        t_end   = track.get("end_time",   0.0)
        col     = COLOURS[cam_idx % len(COLOURS)]

        if   len(flist) == 0:            sampled = [None]*3
        elif len(flist) <= FRAMES_PER_CAM: sampled = flist
        else:
            step = len(flist)//FRAMES_PER_CAM
            sampled = flist[::step][:FRAMES_PER_CAM]

        for f_str in sampled:
            canvas = np.zeros((ch, cw, 3), dtype=np.uint8)
            cv2.rectangle(canvas, (0,0), (cw,TITLE_H), (18,18,18), -1)
            put_label(canvas, f"Global ID: {global_id}  |  {n_cams} cams  |  Active: {cam_id}  {t_start:.0f}s→{t_end:.0f}s",
                      8, 26, (230,230,230), 0.50)
            for i, t in enumerate(tracks):
                ci      = t.get("camera_id", f"cam{i}")
                fi_list = t.get("frames", [])
                cicol   = COLOURS[i % len(COLOURS)]
                is_act  = (i == cam_idx)
                row_i, col_i = i//cols, i%cols
                xo = GAP + col_i*(PANEL_W+GAP)
                yo = TITLE_H + GAP + row_i*(PANEL_H+GAP)
                if is_act:
                    panel = load_panel(f_str, cicol, True)
                    put_label(panel, f"ACTIVE:{ci}", 4, 18, cicol, 0.42)
                    put_label(panel, f"{t.get('start_time',0):.0f}s-{t.get('end_time',0):.0f}s", 4, 35, (200,200,200), 0.38)
                else:
                    panel = load_panel(fi_list[0] if fi_list else None, cicol, False)
                    status = "PAST" if i < cam_idx else "NEXT"
                    put_label(panel, f"{status}:{ci}", 4, 18, cicol, 0.42)
                canvas[yo:yo+PANEL_H, xo:xo+PANEL_W] = panel

            # timeline
            bar = np.zeros((TIMELINE_H, cw, 3), dtype=np.uint8); bar[:] = (25,25,25)
            sw = cw // len(all_cams)
            for i, cam in enumerate(all_cams):
                x1,x2 = i*sw+3,(i+1)*sw-3
                c = COLOURS[i%len(COLOURS)]
                alpha = 1.0 if cam==cam_id else (0.45 if cam in visited else 0.15)
                cv2.rectangle(bar,(x1,4),(x2,TIMELINE_H-18),tuple(int(v*alpha) for v in c),-1)
                cv2.rectangle(bar,(x1,4),(x2,TIMELINE_H-18),c,3 if cam==cam_id else 1)
                cv2.putText(bar,str(cam),(x1+3,TIMELINE_H-5),cv2.FONT_HERSHEY_SIMPLEX,0.35,c,1,cv2.LINE_AA)
            canvas[ch-TIMELINE_H:] = bar
            writer.write(canvas)
        visited.add(cam_id)
    writer.release()
    return True

def to_h264(raw_path, out_path):
    """Re-encode mp4v → H.264 so browsers can play it inline."""
    result = subprocess.run(
        ["ffmpeg", "-y", "-i", str(raw_path),
         "-vcodec", "libx264", "-crf", "28", "-preset", "fast",
         "-pix_fmt", "yuv420p", str(out_path)],
        capture_output=True)
    return out_path.exists()

def show_video_inline(path, width=700):
    """Embed video as base64 HTML so it plays directly in the notebook."""
    data = Path(path).read_bytes()
    b64  = base64.b64encode(data).decode()
    display(HTML(f"""
    <video width="{width}" controls autoplay loop>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>"""))

# ── MAIN ────────────────────────────────────────────────────────────────
print("="*65)
print("  STAGE 4 — TOP-5 MULTI-CAMERA TRAJECTORY VIDEOS")
print("="*65)

multi = [(gid, tracks) for gid, tracks in global_tracklets.items()
         if isinstance(tracks, list) and len(tracks) > 1]
multi.sort(key=lambda x: (len(x[1]), sum(len(t.get("frames",[])) for t in x[1])), reverse=True)

for rank, (gid, tracks) in enumerate(multi[:TOP_N], 1):
    raw_path = VIDEO_DIR / f"globalid_{gid:04d}_top{rank}_raw.mp4"
    h264_path = VIDEO_DIR / f"globalid_{gid:04d}_top{rank}_{len(tracks)}cams.mp4"

    print(f"\n[{rank}/{TOP_N}] Global ID {gid} | {len(tracks)} cameras")
    ok = render_video(gid, tracks, raw_path)
    if not ok:
        print("  ✗ render failed"); continue

    if to_h264(raw_path, h264_path):
        raw_path.unlink(missing_ok=True)           # delete raw
        kb = h264_path.stat().st_size // 1024
        print(f"  ✓ {h264_path.name}  ({kb} KB)")
        shutil.copy(h264_path, Path("/kaggle/working") / h264_path.name)
        show_video_inline(h264_path)               # ← plays INLINE in notebook
    else:
        print("  ⚠ ffmpeg failed, showing raw (may not play in browser)")
        show_video_inline(raw_path)

print("\n✅ Done — videos in /kaggle/working/stage4/demo_videos/")
